# Test GPT Model
Fill in one TODO in `src/model.py`, then run the corresponding cell to verify it works.

In [1]:
import sys
sys.path.extend(["..", "../src"])   # add project root and src/ to path

import torch
from transformers import AutoTokenizer

from config import GPTConfig
from model import GPT, count_parameters, CausalSelfAttention, MLP, TransformerBlock

cfg = GPTConfig()
print(f"d_model={cfg.d_model}, n_layers={cfg.n_layers}, n_heads={cfg.n_heads}")

d_model=256, n_layers=6, n_heads=8


---
## Test 1: MLP

In [2]:
# fill in MLP first, then run this
mlp = MLP(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = mlp(x)
print(f"MLP forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"

MLP forward OK: [2, 8, 256] -> [2, 8, 256]


---
## Test 2: CausalSelfAttention

In [3]:
# fill in attention forward, then run this
attn = CausalSelfAttention(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = attn(x)
print(f"Attention forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"
print("Causal mask works: all good")

Attention forward OK: [2, 8, 256] -> [2, 8, 256]
Causal mask works: all good


---
## Test 3: TransformerBlock

In [4]:
# fill in TransformerBlock, then run this
block = TransformerBlock(cfg)
x = torch.randn(2, 8, cfg.d_model)
y = block(x)
print(f"Block forward OK: {list(x.shape)} -> {list(y.shape)}")
assert y.shape == x.shape, f"shape mismatch {y.shape} != {x.shape}"

Block forward OK: [2, 8, 256] -> [2, 8, 256]


---
## Test 4: Full GPT (forward + loss)

In [5]:
# fill in GPT __init__ and forward, then run this
model = GPT(cfg)
n_params = count_parameters(model)
print(f"Total params: {n_params:.2f}M")
assert abs(n_params - 1.2) < 0.5, f"unexpected param count: {n_params:.2f}M"

# forward with loss
x = torch.randint(0, cfg.vocab_size, (2, 64))
logits, loss = model(x, targets=x)
print(f"Forward OK: logits {list(logits.shape)}, loss {loss.item():.4f}")
assert logits.shape == (2, 64, cfg.vocab_size), f"logits shape wrong: {logits.shape}"
assert loss.item() > 0, "loss should be positive"

Total params: 30.59M


AssertionError: unexpected param count: 30.59M

---
## Test 5: Generation

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

prompt = tokenizer("Once upon a time", return_tensors="pt")["input_ids"]
output = model.generate(prompt, max_new_tokens=30, temperature=1.0)
generated = tokenizer.decode(output[0])
print("Generated:")
print(generated)